<a href="https://colab.research.google.com/github/ivasylenko1/research_seminar_cv_search_project/blob/main/CV_embedding_finetune_from_pairs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning з уже готових train_pairs.csv / auto_eval_pool.csv

Без Section 9 (relabeling) — використовуємо файли, які вже є на Drive.

**Що ймовірно зламало попередню модель:** якщо тренування переривалось через
OOM і ти після того завантажував модель з `output_dir/checkpoint-XXX`
(проміжний checkpoint від `Trainer`) замість фінальної збереженої моделі —
такий checkpoint не має `modules.json`/pooling-конфігурації, яку очікує
`SentenceTransformer`, і `.encode()` або падає, або повертає сміття.

Ця версія:
- тренує з безпечними для T4 налаштуваннями (без багів у назвах аргументів/імпортах)
- зберігає модель ЯВНО одним викликом `model.save(output_dir)` після успішного `trainer.train()`
- одразу після збереження **перезавантажує модель у свіжу змінну і перевіряє, що `.encode()` реально працює**

### 1. Setup

In [ ]:
# --- Setup: мінімум, що реально потрібен для тренування ---

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # має стояти ДО import torch

import gc
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator

from google.colab import drive
drive.mount("/content/drive")

CONFIG = {
    "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
    "output_dir": "/content/drive/MyDrive/qwen-cv-retriever-finetuned-v2",  # НОВА папка, не старий output_dir
    "train_pairs_path": "/content/drive/MyDrive/train_pairs.csv",
    "auto_eval_pool_path": "/content/drive/MyDrive/auto_eval_pool.csv",
}
DOC_PROMPT = "Represent this candidate CV for job matching retrieval"
QUERY_PROMPT = "Find candidates matching this job vacancy"

gc.collect()
torch.cuda.empty_cache()

embed_model = SentenceTransformer(CONFIG["embedding_model"])  # без torch_dtype=float16 —
# на A100 тренуємось у bf16 через autocast; примусове float16 тут і fp16-AMP разом
# спричиняють "Attempting to unscale FP16 gradients"
embed_model.max_seq_length = 512  # безпечніше за 1024 для T4 — знижує шанс OOM
print(f"Модель завантажена: {CONFIG['embedding_model']}, max_seq_length={embed_model.max_seq_length}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'немає'}")


/tmp/ipykernel_485/3383061729.py:11: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_485/3383061729.py:17: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Модель завантажена: Qwen/Qwen3-Embedding-0.6B, max_seq_length=512
GPU: NVIDIA A100-SXM4-40GB


### 2. Training data з `train_pairs.csv`

In [ ]:
# --- Завантаження train_pairs.csv ---

train_pairs_df = pd.read_csv(CONFIG["train_pairs_path"])
print(f"train_pairs.csv: {len(train_pairs_df)} рядків, колонки: {list(train_pairs_df.columns)}")

anchors = (QUERY_PROMPT + ": " + train_pairs_df["vacancy_text"]).tolist()
positives = (DOC_PROMPT + ": " + train_pairs_df["cv_text_positive"]).tolist()
negatives = (DOC_PROMPT + ": " + train_pairs_df["cv_text_hard_negative"]).tolist()

train_dataset = Dataset.from_dict({"anchor": anchors, "positive": positives, "negative": negatives})
print(f"train_dataset готовий: {len(train_dataset)} прикладів")


train_pairs.csv: 1209 рядків, колонки: ['vacancy_text', 'cv_text_positive', 'cv_text_hard_negative']
train_dataset готовий: 1209 прикладів


### 3. Eval data з `auto_eval_pool.csv`

Автоматичне визначення назв колонок (файл міг зберегтись з трохи іншими
іменами колонок, ніж очікувалось спочатку).

In [ ]:
# --- Завантаження auto_eval_pool.csv з авто-визначенням колонок ---

eval_df = pd.read_csv(CONFIG["auto_eval_pool_path"])
print("Колонки у файлі:", list(eval_df.columns))

def find_column(df, keywords, default_idx=0):
    for col in df.columns:
        if any(kw in col.lower() for kw in keywords):
            return col
    return df.columns[default_idx]

vacancy_title_col = find_column(eval_df, ["vacancy_title", "title", "vacancy", "job_title"])
vacancy_text_col = find_column(eval_df, ["vacancy_text", "description", "descr", "embed_text"])
cv_text_col = find_column(eval_df, ["cv_text", "cv", "resume", "candidate_text"])
label_col = find_column(eval_df, ["auto_score", "score", "label", "rating"])

print(f"vacancy_title -> {vacancy_title_col!r}, vacancy_text -> {vacancy_text_col!r}, "
      f"cv_text -> {cv_text_col!r}, label -> {label_col!r}")

# Якщо у файлі нема окремого vacancy_text (тільки vacancy title), групуємо по назві
group_col = vacancy_title_col

eval_data = {}
for title, group in eval_df.groupby(group_col):
    title_str = str(title)
    v_text = group[vacancy_text_col].iloc[0] if vacancy_text_col in group.columns else title_str
    eval_data[title_str] = {
        "vacancy_text": str(v_text),
        "cv_texts": [str(t) for t in group[cv_text_col].tolist()],
        "labels": pd.to_numeric(group[label_col], errors="coerce").fillna(0.0).tolist(),
    }

print(f"eval_data: {len(eval_data)} вакансій")

# labels тут — auto_score від Qwen3-Reranker (0..1), а не людська шкала 0-3,
# тому поріг для evaluator має бути float, не None
EVAL_POSITIVE_THRESHOLD = 0.70  # узгодь з тим порогом, яким генерувався auto_eval_pool.csv


Колонки у файлі: ['vacancy', 'cv_text', 'auto_score']
vacancy_title -> 'vacancy', vacancy_text -> 'vacancy', cv_text -> 'cv_text', label -> 'auto_score'
eval_data: 37 вакансій


### 4. Evaluator

In [ ]:
# --- IR Evaluator ---

def build_ir_evaluator(eval_data, positive_threshold):
    queries, corpus, relevant_docs = {}, {}, {}
    for title, data in eval_data.items():
        qid = title
        queries[qid] = QUERY_PROMPT + ": " + data["vacancy_text"]
        relevant_docs[qid] = set()
        for i, (cv_text, label) in enumerate(zip(data["cv_texts"], data["labels"])):
            cid = f"{title}__{i}"
            corpus[cid] = DOC_PROMPT + ": " + cv_text
            if label >= positive_threshold:
                relevant_docs[qid].add(cid)
    return InformationRetrievalEvaluator(
        queries=queries, corpus=corpus, relevant_docs=relevant_docs,
        name="cv-vacancy-eval",
        precision_recall_at_k=[1, 3, 5, 10],
        mrr_at_k=[10], ndcg_at_k=[10],
        batch_size=8,
    )

evaluator = build_ir_evaluator(eval_data, EVAL_POSITIVE_THRESHOLD)
print(f"Evaluator готовий: {len(evaluator.queries)} вакансій, {len(evaluator.corpus)} унікальних CV")


Evaluator готовий: 29 вакансій, 1134 унікальних CV


### 5. Loss

In [ ]:
# --- Loss: MultipleNegativesRankingLoss (+ опційно Matryoshka) ---

USE_MATRYOSHKA = True
MATRYOSHKA_DIMS = [1024, 768, 512, 256, 128]

base_loss = losses.MultipleNegativesRankingLoss(embed_model)
train_loss = losses.MatryoshkaLoss(embed_model, base_loss, matryoshka_dims=MATRYOSHKA_DIMS) if USE_MATRYOSHKA else base_loss
print("Loss готовий")


Loss готовий


### 6. Тренування (A100)

`bf16` замість `fp16` — на A100 це нативно підтримується і не конфліктує з
`GradScaler`. Батч 8 + gradient accumulation ×4 (ефективний батч 32) +
gradient checkpointing — щоб реально влізти в пам'ять: повний батч 32 без
checkpointing переповнював навіть A100 40GB через MultipleNegativesRankingLoss
(кожен приклад = anchor+positive+negative, тобто 32×3=96 послідовностей за крок).

In [ ]:
# --- Тренування ---

args = SentenceTransformerTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=3,
    per_device_train_batch_size=8,        # зменшено з 32 — 32 переповнював навіть A100
    gradient_accumulation_steps=4,        # ефективний батч лишається 32
    per_device_eval_batch_size=8,
    gradient_checkpointing=True,          # економить пам'ять на активаціях (безпечно з bf16)
    learning_rate=2e-5,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    bf16=True,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="cv-vacancy-eval_cosine_ndcg@10",  # evaluator не рахує eval_loss —
    greater_is_better=True,                                   # треба вказати його власну метрику явно
    dataloader_num_workers=2,
)

trainer = SentenceTransformerTrainer(
    model=embed_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

trainer.train()

# Явне збереження ФІНАЛЬНОЇ моделі у sentence-transformers форматі
# (НЕ бери model з output_dir/checkpoint-XXX — це проміжні чекпоінти Trainer,
# бери саме CONFIG["output_dir"] напряму, куди зараз зберігаємо)
embed_model.save(CONFIG["output_dir"])
print(f"Модель збережена → {CONFIG['output_dir']}")


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Cv-vacancy-eval Cosine Accuracy@1,Cv-vacancy-eval Cosine Accuracy@3,Cv-vacancy-eval Cosine Accuracy@5,Cv-vacancy-eval Cosine Accuracy@10,Cv-vacancy-eval Cosine Precision@1,Cv-vacancy-eval Cosine Precision@3,Cv-vacancy-eval Cosine Precision@5,Cv-vacancy-eval Cosine Precision@10,Cv-vacancy-eval Cosine Recall@1,Cv-vacancy-eval Cosine Recall@3,Cv-vacancy-eval Cosine Recall@5,Cv-vacancy-eval Cosine Recall@10,Cv-vacancy-eval Cosine Ndcg@10,Cv-vacancy-eval Cosine Mrr@10,Cv-vacancy-eval Cosine Map@100
1,4.494548,No log,0.241379,0.275862,0.448276,0.551724,0.241379,0.149425,0.158621,0.120690,0.070197,0.123974,0.218391,0.314039,0.250954,0.305788,0.215982
2,1.423447,No log,0.206897,0.344828,0.413793,0.586207,0.206897,0.172414,0.144828,0.120690,0.052956,0.152709,0.201970,0.320608,0.251413,0.310482,0.212491
3,1.075774,No log,0.241379,0.310345,0.448276,0.586207,0.241379,0.160920,0.158621,0.117241,0.070197,0.141215,0.224959,0.315681,0.257107,0.326232,0.223372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель збережена → /content/drive/MyDrive/qwen-cv-retriever-finetuned-v2


### 7. Перевірка: модель дійсно видає ембеддінги?

Це саме той крок, якого не вистачало минулого разу — перевіряємо ОДРАЗУ,
у свіжій змінній (новий `SentenceTransformer(...)`, не той самий `embed_model`
з пам'яті), що збережена модель реально працює.

In [ ]:
# --- Верифікація збереженої моделі ---

del embed_model  # прибираємо стару змінну з пам'яті, щоб точно тестувати ФАЙЛИ, а не живий об'єкт
gc.collect()
torch.cuda.empty_cache()

reloaded_model = SentenceTransformer(CONFIG["output_dir"])
test_vec = reloaded_model.encode("test sentence", normalize_embeddings=True)

print(f"Тип: {type(test_vec)}")
print(f"Форма: {test_vec.shape}")
print(f"Норма (має бути ≈1.0, бо normalize_embeddings=True): {np.linalg.norm(test_vec):.4f}")
print(f"Чи є NaN: {np.isnan(test_vec).any()}")

assert test_vec.shape[0] > 0, "Порожній вектор — модель не зберегла pooling/dense шар!"
assert not np.isnan(test_vec).any(), "NaN у векторі — щось зламалось під час тренування!"
print("\n✅ Модель успішно видає ембеддінги очікуваної форми")


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Тип: <class 'numpy.ndarray'>
Форма: (1024,)
Норма (має бути ≈1.0, бо normalize_embeddings=True): 1.0008
Чи є NaN: False

✅ Модель успішно видає ембеддінги очікуваної форми


### 8. Порівняння baseline vs fine-tuned на eval_data

In [ ]:
import gc
if 'trainer' in locals():
    del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"Вільно на GPU після очищення: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Вільно на GPU після очищення: 33.92 GB


In [ ]:
def compare_embeddings_ndcg(model, eval_data, k=10):
    rows = []
    for title, data in eval_data.items():
        query_vec = model.encode(data["vacancy_text"], prompt=QUERY_PROMPT, normalize_embeddings=True, batch_size=4)
        doc_vecs = model.encode(data["cv_texts"], prompt=DOC_PROMPT, normalize_embeddings=True, batch_size=4, show_progress_bar=False)
        scores = doc_vecs @ query_vec
        ndcg = ndcg_score(np.array([data["labels"]]), np.array([scores]), k=k)
        rows.append({"vacancy": title[:40], f"NDCG@{k}": round(ndcg, 4)})
    result_df = pd.DataFrame(rows)
    print(result_df.to_string(index=False))
    print(f"Середній NDCG@{k}: {result_df[f'NDCG@{k}'].mean():.4f}")
    return result_df

In [ ]:
print("=== Baseline (оригінальна Qwen3-Embedding, без файн-тюнінгу) ===")
baseline_model = SentenceTransformer(CONFIG["embedding_model"])
baseline_results = compare_embeddings_ndcg(baseline_model, eval_data, k=10)

del baseline_model
gc.collect()
torch.cuda.empty_cache()

print("\n=== Fine-tuned ===")
finetuned_results = compare_embeddings_ndcg(reloaded_model, eval_data, k=10)

=== Baseline (оригінальна Qwen3-Embedding, без файн-тюнінгу) ===


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

                                 vacancy  NDCG@10
                            1C Developer   0.3730
                               2D Artist   0.4483
           Amazon Account Health Manager   0.5048
Assembler / No-code AI Developer for iGa   0.5738
            Back-end (physical) engineer   0.5647
                BizDev Manager (Network)   0.4245
                  Business Analyst (ERP)   0.3939
                    Business Analyst POP   0.6599
            Business development manager   0.5833
                 Chief Executive Officer   0.6521
                                  DevOps   0.7435
Full-Stack Developer (Fintech Backend fo   0.4965
                             Head of CRM   0.3933
               Head of Engineering / CTO   0.6716
                         Head of Product   0.5833
                      In-App Media Buyer   0.6447
         Junior International Accountant   0.5569
                      Junior QA Engineer   0.6912
       Junior technical support engineer   0.7857


In [ ]:
baseline_model.to("cpu")
del baseline_model

---
### Якщо `.encode()` все ще падає після цього

- Перевір вміст `CONFIG["output_dir"]` — там мають бути файли
  `modules.json`, `config_sentence_transformers.json`, `1_Pooling/config.json`,
  `model.safetensors` (або `pytorch_model.bin`). Якщо `modules.json` нема —
  збереження не завершилось (перевір, чи `trainer.train()` дійшов до кінця
  без винятку, і чи вистачило місця на Drive).
- Не бери шлях виду `.../checkpoint-100` для інференсу — це проміжні
  чекпоінти `Trainer`, бери саме `CONFIG["output_dir"]`.